In [ ]:
import os, torch

def _find_file(name, root='/kaggle/input'):
    for r, dirs, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(name)

for plane in ['axial', 'coronal', 'sagittal']:
    path = _find_file(f'{plane}_ema.pt')
    print(f'=== {plane}: {path} ===')
    ck = torch.load(path, map_location='cpu')
    print('  top-level keys:', list(ck.keys()))
    for k, v in ck.items():
        if k == 'ema':
            continue
        print(f'  {k}: {v if not hasattr(v, "shape") else v.shape} (type {type(v)})')
    # checksum of a slice of actual weights, to check if any two files are IDENTICAL
    # (a real bug would be uploading the same file three times under different names)
    sd = ck['ema']
    first_key = list(sd.keys())[0]
    print('  first param key:', first_key, '| shape:', sd[first_key].shape,
          '| sum:', float(sd[first_key].sum()))
    del ck, sd


In [ ]:
# Cross-check: are the three checkpoints actually DIFFERENT models, or accidental duplicates?
import hashlib
def file_hash(path, n=1_000_000):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read(n)).hexdigest()

for plane in ['axial', 'coronal', 'sagittal']:
    path = _find_file(f'{plane}_ema.pt')
    print(plane, file_hash(path))
